# Detekce mutací <i>Plasmodia falciparum</i> pomocí bioinformatických nástrojů v příkazové řádce
## Obsah hodiny


## Motivace: Proč nás zajímají mutace plasmodia?
TODO: nějaký obrázek, malárie, rezistence, geny


## Co je to příkazová řádka?

<b>GUI = graphic user interface = grafické rozhraní</b>

Ovládání počítače pomocí interaktivních grafických prvků jako například okna, menu, klikací ikony

<b>CLI = command line interface = příkazová řádka (terminál) </b>

Ovládání počítače pomocí textových příkazů 

<details>
  <summary>Proč používat příkazovou řádku pro řešení bioinformatických úkolů?</summary>
   
+ flexibilita v kombinování existujících nástrojů a možnost zapojení vlastních skriptů
+ zpracování velkého množství souborů najednou
+ zpracování velkých souborů a náročných výpočtů na vzdálených serverech, kde GUI není dostupné
+ snadné zopakování analýzy kdykoli a kýmkoli  

</details>

<br/><br/>
<b>Jak otevřít příkazovou řádku v Jupyter Lab</b>

V horním menu klikněte na <b>File</b> > <b>New</b> > <b>Terminal</b>

Vyzkoušejte zadat příkaz do příkazové řádky (potvrdít stisknutím <b>Enter<b>):

- `ls` - vypsat soubory v aktuálním adresáři
- `pwd` - vypsat cestu do aktuálního adresáře
- `date` - vypsat aktuální systémový čas

Nebo můžete spustit příkaz přímo v buňkách s kódem v tomto notebooku:  

In [ ]:
!ls 
# označte tuto buňku šipkami na klávesnici (nebo levým tlačítkem myši) a stiskněte Shift+Enter na klávesnici (nebo ikonu "Run this cell and advance" na horní liště)
# můžete taky obsah buňky přepsat - vyzkoušejte, jak se výstup příkazu změní, když u něj bude doplňkový parametr - jako například ls -la
# hashtag (#) signalizuje, že na řádku není zapsaný kód ke spuštění, ale komentář

<b>Poznámka:</b> `!` na začátku buňky říká Jupyter Labu "spusť kód jako příkaz v terminálu". Když do buňky napíšete kód bez vykřičníku: `print("ahoj")`, poběží jako python kód. Příkazová řádka a python používají jinou "gramatiku" takže je potřeba odlišit, jak se má příkaz spustit. My budeme dále posílat příkazy pouze do terminálu, takže pokud budete spouštět příkazy v buňkách JupyterLab Notebooku, všechny by měly začínat `!`. 

In [ ]:
print("ahoj pythone!")
!echo "ahoj příkazová řádko!"

## Získání vstupních souborů
potřebujeme: 
- sekvence DNA plasmodia z pacientů - krátká čtení
- referenční sekvenci plasmodia - DNA sekvence celého genomu 
- pozici genu <i>dhfr</i> v referenční sekvenci (tzv. genové anotace)

Všechny potřebné soubory pro toto cvičení (původně vytvořeny pro [tuto Galaxy lekci](https://training.galaxyproject.org/training-material/topics/introduction/tutorials/galaxy-intro-ngs-data-managment/tutorial.html#annotating-variants{target=_blank})) jsou dostupné v internetovém archivu, ze kterého si je můžeme stáhnout. Začátek příkazu (`wget`) určuje, jaký program se má spustit, pak následuje URL adresa souboru, který chceme stáhnout.

In [ ]:
!wget https://zenodo.org/records/15354240/files/GCF_000002765.6.fa.gz # referenční sekvence plasmodium falciparum - formát fasta
!wget https://zenodo.org/records/15354240/files/GCF_000002765.6_GCA_000002765.ncbiRefSeq.gtf.gz # genové anotace - formát gtf
!wget https://zenodo.org/records/15354240/files/ERR042228_F.fq.gz # soubor obsahující DNA čtení (DNA reads)  - formát fastq
!wget https://zenodo.org/records/15354240/files/ERR042228_R.fq.gz # soubor obsahující DNA čtení (DNA reads) - formát fastq


Stažené soubory mají příponu `.gz`, jsou tedy komprimované za účelem zmenšení jejich velikosti. Abychom s nimi mohli snadno pracovat, je nutné je nejdřív rozbalit pomocí `gunzip`.

<b>Poznámka:</b> `*` je takzvaný "žolíkový znak" (wildcard), který zastupuje libovolné znaky. Takže příkaz níže rozbalí všechny soubory uvnitř aktuální složky, které končí `.gz` a není potřeba vypisovat názvy jednotlivých souborů.


In [ ]:
!gunzip -f *.gz

<b>Poznámka:</b> Parametry připojené za příkazem modifikují jeho chování. Pokud se chcete podívat, co znamená parametr `-f`, podívejte se na nápovědu k příkazu pomocí `gunzip --help`.  

Nyní se můžeme podívat na obsah stažených souborů pomocí `head`, který nám vypíše prvních 10 řádků souboru.
Začneme s referenční sekvencí, která obsahuje celý genom plasmodia. 

In [ ]:
!head GCF_000002765.6.fa

Vidíme, že soubor obsahuje sekvenci nukleotidů (kombinace písmenek ATCG). První řádek ale obsahuje `>` a následně název sekvence. Genom plasmodia totiž obsahuje několik chromozomů, jejichž sekvence jsou v tomto souboru zapsané za sebou.
Názvy chromozomů a číslo řádku na kterém začínají si můžete vypsat, když pomocí příkazu `grep` vyhledáme v souboru řádky obsahující `>`.

In [ ]:
!grep -n ">" GCF_000002765.6.fa

Dále se můžeme podívat na soubory s DNA čteními, ve kterých budeme hledat mutace.

In [ ]:
!head ERR042228_F.fq

Tento soubor také zachycuje DNA sekvence, ale oproti předchozímu souboru obsahuje informace navíc. Na prvním řádku je opět název, na druhém samotná sekvence nukleotidů a na čtvrtém je kvalita čtení jednotlivých nukleotidů ve formě řady ASCII znaků. Začátky a konce DNA čtení často mají horší kvalitu, a tak je možné tuto informaci využít k filtrování čtení či částí čtení s nízkou kvalitou.

<b>Poznámka:</b> Stáhli jsme si dva soubory DNA čtení pro každý vzorek, protože sekvenování bylo provedeno metodou, která přečte zhruba 100 nukleotidů z obou konců většího kusu DNA (~300 nukleotidů). Pro každý kus tak máme <b>F</b>orward a <b>R</b>everse sekvenci.  

Poslední soubor vyjadřuje pozice genů v referenčním genomu.

In [11]:
!head GCF_000002765.6_GCA_000002765.ncbiRefSeq.gtf

NC_037283.1	ncbiRefSeq.2024-10-03	transcript	3290953	3291501	.	+	.	gene_id "PF3D7_1480100"; transcript_id "PF3D7_1480100";  gene_name "PF3D7_1480100";
NC_037283.1	ncbiRefSeq.2024-10-03	exon	3290953	3291501	.	+	.	gene_id "PF3D7_1480100"; transcript_id "PF3D7_1480100"; exon_number "1"; exon_id "PF3D7_1480100.1"; gene_name "PF3D7_1480100";
NC_037283.1	ncbiRefSeq.2024-10-03	transcript	3285900	3287003	.	+	.	gene_id "PF3D7_1480000"; transcript_id "XM_001348909.1";  gene_name "PF3D7_1480000";
NC_037283.1	ncbiRefSeq.2024-10-03	exon	3285900	3285953	.	+	.	gene_id "PF3D7_1480000"; transcript_id "XM_001348909.1"; exon_number "1"; exon_id "XM_001348909.1.1"; gene_name "PF3D7_1480000";
NC_037283.1	ncbiRefSeq.2024-10-03	CDS	3285900	3285953	.	+	0	gene_id "PF3D7_1480000"; transcript_id "XM_001348909.1"; exon_number "1"; exon_id "XM_001348909.1.1"; gene_name "PF3D7_1480000";
NC_037283.1	ncbiRefSeq.2024-10-03	exon	3286035	3287003	.	+	.	gene_id "PF3D7_1480000"; transcript_id "XM_001348909.1"; exon_number 

Tento soubor je v postatě tabulka (oddělená tabulátorovou mezerou), která obsahuje název genu, název chromozomu, na kterém se daný gen nachází, přesnou numerickou pozici na daném chromozomu a další informace popisující vlastnosti genu, jako například jeho jednotlivé části (exony, introny, kódující sekvence, začátek transkriptu).
<details>
  <summary>Co obsahují jednotlivé sloupce tabulky?</summary>

- seqname - name of the chromosome or scaffold
- source - name of the program that generated this feature, or the data source (database or project name)
- feature - feature type name, e.g. Gene, Variation, Similarity
- start - Start position of the feature, with sequence numbering starting at 1.
- end - End position of the feature, with sequence numbering starting at 1.
- score - A floating point value.
- strand - defined as + (forward) or - (reverse).
- frame - One of '0', '1' or '2'. '0' indicates that the first base of the feature is the first base of a codon, '1' that the second base is the first base of a codon, and so on..
- attribute - A semicolon-separated list of tag-value pairs, providing additional information about each feature.
</details>

Můžeme také v souboru vyhledat gen, který může obsahovat mutaci způsobující rezistenci.

In [22]:
!grep "PF3D7_0417200" GCF_000002765.6_GCA_000002765.ncbiRefSeq.gtf

NC_004318.2	ncbiRefSeq.2024-10-03	transcript	748088	749914	.	+	.	gene_id "PF3D7_0417200"; transcript_id "XM_001351443.1";  gene_name "PF3D7_0417200";
NC_004318.2	ncbiRefSeq.2024-10-03	exon	748088	749914	.	+	.	gene_id "PF3D7_0417200"; transcript_id "XM_001351443.1"; exon_number "1"; exon_id "XM_001351443.1.1"; gene_name "PF3D7_0417200";
NC_004318.2	ncbiRefSeq.2024-10-03	CDS	748088	749911	.	+	0	gene_id "PF3D7_0417200"; transcript_id "XM_001351443.1"; exon_number "1"; exon_id "XM_001351443.1.1"; gene_name "PF3D7_0417200";
NC_004318.2	ncbiRefSeq.2024-10-03	start_codon	748088	748090	.	+	0	gene_id "PF3D7_0417200"; transcript_id "XM_001351443.1"; exon_number "1"; exon_id "XM_001351443.1.1"; gene_name "PF3D7_0417200";
NC_004318.2	ncbiRefSeq.2024-10-03	stop_codon	749912	749914	.	+	0	gene_id "PF3D7_0417200"; transcript_id "XM_001351443.1"; exon_number "1"; exon_id "XM_001351443.1.1"; gene_name "PF3D7_0417200";


## Mapování sekvencí na referenční genom
Nejdřív si musíme nainstalovat nástroj pro mapování. Ten se nazývá BWA (Burrows-Wheeler Alignment) a funguje na základě hledání krátkých sekvencí v DNA readech, které se přesně shodují referencí. Aby bylo toto hledání rychlé, nástroj nejříve DNA sekvenci algoritmicky přetransformuje do podoby, ve které se tyto podobnosti hledají snáz - takzvaného indexu.

<b>Poznámka:</b> K instalaci nástrojů používáme jiný nástroj - mamba. Tento oblíbený software umožňuje snadno instalovat bioinformatické nástroje, a to i v případě, že nástroj ke své funkci potřebuje další nástroje od jiných tvůrců. Umožňuje tak programátorům stavět na existujících balíčcích volně dostupných bioinformatických nástrojů.       

In [ ]:
!mamba install bioconda::bwa --yes

In [17]:
!bwa index GCF_000002765.6.fa # nejdříve vytvoříme index z referenční sekvence
!bwa mem GCF_000002765.6.fa ERR042228_F.fq ERR042228_R.fq > ERR042228_mapping.sam # potom mapujeme DNA ready na referenční sekvenci, výsledek uložíme do souboru ERR042228_mapping.sam

[bwa_index] Pack FASTA... 0.10 sec
[bwa_index] Construct BWT for the packed sequence...
[bwa_index] 6.24 seconds elapse.
[bwa_index] Update BWT... 0.08 sec
[bwa_index] Pack forward-only FASTA... 0.05 sec
[bwa_index] Construct SA from BWT and Occ... 4.54 sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa index GCF_000002765.6.fa
[main] Real time: 12.605 sec; CPU: 11.022 sec
[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 128606 sequences (9641628 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (0, 59290, 0, 0)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (220, 261, 314)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (32, 502)
[M::mem_pestat] mean and std.dev: (271.12, 69.74)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 596)
[M::mem_pestat] skip orientation RF as there are not 

## Hledání mutací

Nejdříve musíme připravit soubor namapovaných readů do binárního formátu.

In [25]:
!mamba install bioconda::samtools --yes # nainstalovat nástroj na konverzi formátu sam -> bam
!mamba install bioconda::lofreq --yes # nainstalovat nástroj na heldání mutací

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache

Pinned packages:

  - python=3.12

Pinned packages:

  - python=3.12


Transaction

  Prefix: /opt/conda

  All requested packages already installed


Transaction starting

Transaction finished

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache

Pinned packages:

  - python=3.12

Pinned packages:

  - python=3.12


Transaction

  Prefix: /opt/conda

  Updating specs:

   - bioconda::lofreq


  Package     Version  Build             Channel           Size
──────────────

In [32]:
!samtools view -b ERR042228_mapping.sam > ERR042228_mapping.bam # konvertovat soubor do vhodného formátu
!samtools sort ERR042228_mapping.bam -o ERR042228_mapping.sorted.bam # seřadit namapovaná čtení podle jejich pozice v genomu pro efektivnějí 
!samtools faidx GCF_000002765.6.fa
!samtools index ERR042228_mapping.sorted.bam
!lofreq indelqual --dindel -f GCF_000002765.6.fa -o ERR042228_mapping.indelqual.bam ERR042228_mapping.sorted.bam # opravit chybně namapovaná čtení kolem inzercí a delecí
!lofreq call -f GCF_000002765.6.fa --call-indels --min-cov 10 --min-bq 20 --min-alt-bq 20 -q 20 -o ERR042228_variants.vcf ERR042228_mapping.indelqual.bam # najít mutace v namapovaných čteních

WARNING(lofreq_call.c|main_call): 4 indel calls (before filtering) were made without indel alignment-quality! Did you forget to indel alignment-quality to your bam-file?
Number of substitution tests performed: 46944
Number of indel tests performed: 1974


Nyní se můžeme podívat na nalezené mutace. Na začátku souboru je hlavička, která obsahuje informace o souboru. Můžeme se jí vyhnout pomocí příkazu `grep` a vypsat prvních 10 řádků souboru, které neobsahují znak `#`.

In [34]:
!grep -v "#" ERR042228_variants.vcf | head

NC_037280.1	259443	.	T	A	166	PASS	DP=16;AF=1.000000;SB=0;DP4=0,0,11,5
NC_004318.2	657697	.	T	C	324	PASS	DP=14;AF=0.857143;SB=0;DP4=1,0,13,0
NC_004318.2	657869	.	AATAT	A	88	PASS	DP=30;AF=0.100000;SB=0;DP4=5,6,2,1;INDEL;HRUN=4
NC_004318.2	657869	.	AAT	A	510	PASS	DP=30;AF=0.500000;SB=0;DP4=5,6,8,7;INDEL;HRUN=4
NC_004318.2	658024	.	C	CA	74	PASS	DP=58;AF=0.103448;SB=1;DP4=27,25,2,4;INDEL;HRUN=12
NC_004318.2	658212	.	T	TAATA	388	PASS	DP=42;AF=0.309524;SB=0;DP4=13,17,6,7;INDEL;HRUN=2
NC_004318.2	658238	.	AATATATATAT	A	341	PASS	DP=30;AF=0.333333;SB=5;DP4=12,13,2,8;INDEL;HRUN=4
NC_004318.2	658310	.	TA	T	58	PASS	DP=38;AF=0.131579;SB=4;DP4=18,15,4,1;INDEL;HRUN=15
NC_004318.2	658447	.	A	G	453	PASS	DP=32;AF=0.718750;SB=11;DP4=5,1,10,16
NC_004318.2	658569	.	ATATATATG	A	190	PASS	DP=35;AF=0.371429;SB=3;DP4=10,12,8,5;INDEL;HRUN=1
grep: write error: Broken pipe


Soubor obsahuje pozici mutace, který nukleotid se změnil a jak, jaká je kvalita a počet čtení v tomto místě.
<details>
  <summary>Co obsahují jednotlivé sloupce tabulky?</summary>

- CHROM	The name of the sequence (typically a chromosome) on which the variation is being called. This sequence is usually known as 'the reference sequence', i.e. the sequence against which the given sample varies.
- POS	The 1-based position of the variation on the given sequence.
- ID	The identifier of the variation, e.g. a dbSNP rs identifier, or if unknown a ".". Multiple identifiers should be separated by semi-colons without white-space.
- REF	The reference base (or bases in the case of an indel) at the given position on the given reference sequence.
- ALT	The list of alternative alleles at this position.
- QUAL	A quality score associated with the inference of the given alleles.
- FILTER	A flag indicating which of a given set of filters the variation has failed or PASS if all the filters were passed successfully.
- INFO    	An extensible list of key-value pairs (fields) describing the variation. See below for some common fields. Multiple fields are separated by semicolons with optional values in the format: <key>=<data>[,data].

</details>

## Obsahuje gen <i>dhfr</i> mutace?

In [36]:
!mamba install bioconda::bedtools --yes

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache

Pinned packages:

  - python=3.12

Pinned packages:

  - python=3.12


Transaction

  Prefix: /opt/conda

  Updating specs:

   - bioconda::bedtools


  Package     Version  Build       Channel      Size
──────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────

  + bedtools   2.31.1  h13024bc_3  bioconda      2MB

  Summary:

  Install: 1 packages

  Total download: 2MB

──────────────────────────────────────────────────────



Transaction starting
[+] 0.0s
Extracting       ━━━━━━━━━━━━━━━━━━━━━━━       0                            0.0s[+] 0.1s
Extracting       ━━━━━━━━━━━━━━━━━━━━━━━       0                            0.0s[+] 0.2s
Extract

## Vizualizace mutací

In [21]:
!samtools faidx GCF_000002765.6.fa
!samtools index ERR042228_mapping.sorted.bam

## Acknowledgement
tvůrci původního GT